In [2]:
import numpy as np
import re
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
import networkx as nx

class TextRankSummarizer:
    def __init__(self):
        # 한국어 형태소 분석기 초기화
        self.okt = Okt()

    def get_nouns(self, text):
        """문장에서 명사만 추출 (2글자 이상)"""
        nouns = self.okt.nouns(text)
        return [n for n in nouns if len(n) > 1]

    def summarize(self, text, top_n=3):
        # 1. 문장 분리 (정규식 사용)
        sentences = re.split(r'(?<=[.!?])\s+', text.strip())
        if len(sentences) <= top_n:
            return sentences

        # 2. TF-IDF를 이용한 문장 벡터화
        # 각 문장을 명사들의 집합으로 변환
        sent_noun_lists = [" ".join(self.get_nouns(s)) for s in sentences]
        
        # 문장 간 유사도를 구하기 위해 TF-IDF 행렬 생성
        tfidf = TfidfVectorizer()
        tfidf_matrix = tfidf.fit_transform(sent_noun_lists)
        
        # 3. 문장 간 유사도 행렬 생성 (Cosine Similarity)
        # 행렬 곱을 통해 모든 문장 쌍 간의 유사도를 한 번에 계산
        similarity_matrix = (tfidf_matrix * tfidf_matrix.T).toarray()

        # 4. 그래프 생성 및 PageRank 알고리즘 적용
        # 유사도 행렬을 그래프의 인접 행렬로 간주
        nx_graph = nx.from_numpy_array(similarity_matrix)
        scores = nx.pagerank(nx_graph) # 각 문장의 중요도(점수) 계산

        # 5. 점수가 높은 상위 N개 문장 인덱스 추출
        top_indices = sorted(scores, key=scores.get, reverse=True)[:top_n]
        
        # 6. 원래 글의 흐름을 유지하기 위해 인덱스 순서대로 정렬
        top_indices.sort()

        return [sentences[i] for i in top_indices]

# --- 실행 예시 ---
if __name__ == "__main__":
    content = """한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우하는 ‘물리법칙’을 스스로 이해, 구조까지 예측하는 AI 모델 ‘리만 확산 모델(R-DM)’을 개발했다고 10일 밝혔다. 이 모델은 분자의 에너지를 직접 고려한다는 점에서 차별적이라는 설명이다. 기존 AI가 분자의 모양을 단순히 흉내내는 차원이었다면, R-DM은 분자 내부에서 어떤 힘이 작용하는지를 고려해 스스로 구조를 다듬을 수 있다는 설명이다. 특히, 분자구조의 에너지가 높을수록 ‘언덕’으로, 낮을수록 ‘골짜기’로 표현하는 등 알기 쉽게 지도 형태로 나타낸다고 전했다. 이를 통해 AI가 에너지가 가장 낮은 골짜기를 찾아 이동할 수 있도록 설계한 것이다. 이는 수학 이론인 리만 기하학을 적용한 것으로, AI가 화학 기본 원리인 “물질은 에너지가 가장 낮은 상태를 선호한다”라는 법칙을 학습한 결과다. 실험 결과, R-DM은 기존 AI보다 최대 20배 이상 높은 정확도를 보였다. 예측 오차는 정밀 양자역학 계산과 거의 차이가 없는 수준까지 줄었다는 설명이다. 이는 AI 분자 구조 예측 기술 중 세계 최고 수준 성능이라고 강조했다. 이 기술을 활용하면 신약 개발은 물론, 차세대 배터리 소재와 고성능 촉매 설계 등에 도움이 될 것이라는 설명이다. 많은 시간이 소요되던 분자 설계 과정을 크게 단축해 연구 개발 속도를 획기적으로 높여주는 ‘AI 시뮬레이터’로 작용한다고 전했다. 더불어, 화학 사고나 유해 물질 확산처럼 실험이 어려운 상황에서도 화학 반응 경로를 빠르게 예측할 수 있어서 환경 및 안전 분야에서 활용 가능성이 크다고 강조했다. 김우연 KAIST 교수는 “AI가 화학의 기본 원리를 이해하고 분자 안정성을 스스로 판단한 첫 사례”라며 “신소재 개발 방식을 근본적으로 바꿀 수 있는 기술”이라고 말했다."""

    summarizer = TextRankSummarizer()
    summary = summarizer.summarize(content, 5)

    print("\n[TextRank 요약 결과]")
    print("-" * 60)
    for i, line in enumerate(summary):
        print(f"{i+1}. {line}")
    print("-" * 60)


[TextRank 요약 결과]
------------------------------------------------------------
1. 한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우하는 ‘물리법칙’을 스스로 이해, 구조까지 예측하는 AI 모델 ‘리만 확산 모델(R-DM)’을 개발했다고 10일 밝혔다.
2. 이 모델은 분자의 에너지를 직접 고려한다는 점에서 차별적이라는 설명이다.
3. 기존 AI가 분자의 모양을 단순히 흉내내는 차원이었다면, R-DM은 분자 내부에서 어떤 힘이 작용하는지를 고려해 스스로 구조를 다듬을 수 있다는 설명이다.
4. 이는 AI 분자 구조 예측 기술 중 세계 최고 수준 성능이라고 강조했다.
5. 김우연 KAIST 교수는 “AI가 화학의 기본 원리를 이해하고 분자 안정성을 스스로 판단한 첫 사례”라며 “신소재 개발 방식을 근본적으로 바꿀 수 있는 기술”이라고 말했다.
------------------------------------------------------------


In [4]:
import numpy as np
import re
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
import networkx as nx

class TextRankSummarizer:
    def __init__(self):
        self.okt = Okt()

    def get_nouns(self, text):
        """명사 추출 (2글자 이상)"""
        nouns = self.okt.nouns(text)
        return [n for n in nouns if len(n) > 1]

    def summarize(self, text, top_n=3):
        # 1. 문장 분리
        sentences = re.split(r'(?<=[.!?])\s+', text.strip())
        if len(sentences) <= top_n:
            return sentences, sentences

        # 2. 전처리 및 TF-IDF 벡터화
        sent_noun_lists = [" ".join(self.get_nouns(s)) for s in sentences]
        tfidf = TfidfVectorizer()
        tfidf_matrix = tfidf.fit_transform(sent_noun_lists)
        
        # 3. 유사도 행렬 생성 및 PageRank 계산
        similarity_matrix = (tfidf_matrix * tfidf_matrix.T).toarray()
        nx_graph = nx.from_numpy_array(similarity_matrix)
        scores = nx.pagerank(nx_graph)

        # --- A. 중요도 순위별 요약 (Ranking Order) ---
        # (점수, 원래 인덱스) 쌍을 만들어 점수 내림차순 정렬
        rank_idx_pairs = sorted(((scores[i], i) for i in range(len(sentences))), reverse=True)
        top_rank_indices = rank_idx_pairs[:top_n]
        
        ranking_summary = []
        for rank, (score, idx) in enumerate(top_rank_indices):
            ranking_summary.append({
                "rank": rank + 1,
                "score": score,
                "text": sentences[idx]
            })

        # --- B. 문맥 유지 요약 (Original Order) ---
        # 상위 N개의 인덱스만 뽑아서 다시 번호순(오름차순)으로 정렬
        top_indices = sorted([pair[1] for pair in top_rank_indices])
        context_summary = [sentences[i] for i in top_indices]

        return ranking_summary, context_summary

# --- 실행부 ---
if __name__ == "__main__":
    content = """한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우하는 ‘물리법칙’을 스스로 이해, 구조까지 예측하는 AI 모델 ‘리만 확산 모델(R-DM)’을 개발했다고 10일 밝혔다. 이 모델은 분자의 에너지를 직접 고려한다는 점에서 차별적이라는 설명이다. 기존 AI가 분자의 모양을 단순히 흉내내는 차원이었다면, R-DM은 분자 내부에서 어떤 힘이 작용하는지를 고려해 스스로 구조를 다듬을 수 있다는 설명이다. 특히, 분자구조의 에너지가 높을수록 ‘언덕’으로, 낮을수록 ‘골짜기’로 표현하는 등 알기 쉽게 지도 형태로 나타낸다고 전했다. 이를 통해 AI가 에너지가 가장 낮은 골짜기를 찾아 이동할 수 있도록 설계한 것이다. 이는 수학 이론인 리만 기하학을 적용한 것으로, AI가 화학 기본 원리인 “물질은 에너지가 가장 낮은 상태를 선호한다”라는 법칙을 학습한 결과다. 실험 결과, R-DM은 기존 AI보다 최대 20배 이상 높은 정확도를 보였다. 예측 오차는 정밀 양자역학 계산과 거의 차이가 없는 수준까지 줄었다는 설명이다. 이는 AI 분자 구조 예측 기술 중 세계 최고 수준 성능이라고 강조했다. 이 기술을 활용하면 신약 개발은 물론, 차세대 배터리 소재와 고성능 촉매 설계 등에 도움이 될 것이라는 설명이다. 많은 시간이 소요되던 분자 설계 과정을 크게 단축해 연구 개발 속도를 획기적으로 높여주는 ‘AI 시뮬레이터’로 작용한다고 전했다. 더불어, 화학 사고나 유해 물질 확산처럼 실험이 어려운 상황에서도 화학 반응 경로를 빠르게 예측할 수 있어서 환경 및 안전 분야에서 활용 가능성이 크다고 강조했다. 김우연 KAIST 교수는 “AI가 화학의 기본 원리를 이해하고 분자 안정성을 스스로 판단한 첫 사례”라며 “신소재 개발 방식을 근본적으로 바꿀 수 있는 기술”이라고 말했다."""

    summarizer = TextRankSummarizer()
    rank_res, context_res = summarizer.summarize(content, 5)

    print("\n" + "="*30)
    print("1. 중요도 순위별 요약 (Ranking)")
    print("="*30)
    for item in rank_res:
        print(f"[{item['rank']}위] 점수({item['score']:.4f}): {item['text']}")

    print("\n" + "="*30)
    print("2. 문맥 유지 요약 (Context)")
    print("="*30)
    for i, line in enumerate(context_res):
        print(f"{i+1}. {line}")


1. 중요도 순위별 요약 (Ranking)
[1위] 점수(0.0889): 기존 AI가 분자의 모양을 단순히 흉내내는 차원이었다면, R-DM은 분자 내부에서 어떤 힘이 작용하는지를 고려해 스스로 구조를 다듬을 수 있다는 설명이다.
[2위] 점수(0.0876): 한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우하는 ‘물리법칙’을 스스로 이해, 구조까지 예측하는 AI 모델 ‘리만 확산 모델(R-DM)’을 개발했다고 10일 밝혔다.
[3위] 점수(0.0848): 이 모델은 분자의 에너지를 직접 고려한다는 점에서 차별적이라는 설명이다.
[4위] 점수(0.0820): 이는 AI 분자 구조 예측 기술 중 세계 최고 수준 성능이라고 강조했다.
[5위] 점수(0.0819): 김우연 KAIST 교수는 “AI가 화학의 기본 원리를 이해하고 분자 안정성을 스스로 판단한 첫 사례”라며 “신소재 개발 방식을 근본적으로 바꿀 수 있는 기술”이라고 말했다.

2. 문맥 유지 요약 (Context)
1. 한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우하는 ‘물리법칙’을 스스로 이해, 구조까지 예측하는 AI 모델 ‘리만 확산 모델(R-DM)’을 개발했다고 10일 밝혔다.
2. 이 모델은 분자의 에너지를 직접 고려한다는 점에서 차별적이라는 설명이다.
3. 기존 AI가 분자의 모양을 단순히 흉내내는 차원이었다면, R-DM은 분자 내부에서 어떤 힘이 작용하는지를 고려해 스스로 구조를 다듬을 수 있다는 설명이다.
4. 이는 AI 분자 구조 예측 기술 중 세계 최고 수준 성능이라고 강조했다.
5. 김우연 KAIST 교수는 “AI가 화학의 기본 원리를 이해하고 분자 안정성을 스스로 판단한 첫 사례”라며 “신소재 개발 방식을 근본적으로 바꿀 수 있는 기술”이라고 말했다.


In [2]:
import pandas as pd
import numpy as np
import re
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

class SummaryMachine:
    def __init__(self):
        self.okt = Okt()
        self.vectorizer = TfidfVectorizer()

    def get_nouns(self, text):
        """명사 추출 및 전처리"""
        if not isinstance(text, str): return ""
        nouns = self.okt.nouns(text)
        return " ".join([n for n in nouns if len(n) > 1])

    def split_sentences(self, text):
        """문장 분리"""
        sentences = re.split(r'(?<=[.!?])\s+', str(text).strip())
        return [s for s in sentences if len(s.strip()) > 5]

    # --- 1. TF-IDF 기반 요약 ---
    def tfidf_summary(self, text, top_n=3):
        sentences = self.split_sentences(text)
        if len(sentences) <= top_n: return " ".join(sentences)
        sent_nouns = [self.get_nouns(s) for s in sentences]
        try:
            tfidf_matrix = self.vectorizer.fit_transform(sent_nouns)
            scores = np.array(tfidf_matrix.sum(axis=1)).flatten()
            top_indices = scores.argsort()[-top_n:][::-1]
            top_indices.sort()
            return " ".join([sentences[i] for i in top_indices])
        except: return " ".join(sentences[:top_n])

    # --- 2. TextRank 기반 요약 ---
    def textrank_summary(self, text, top_n=3):
        sentences = self.split_sentences(text)
        if len(sentences) <= top_n: return " ".join(sentences)
        sent_nouns = [self.get_nouns(s) for s in sentences]
        try:
            tfidf_matrix = self.vectorizer.fit_transform(sent_nouns)
            sim_mat = (tfidf_matrix * tfidf_matrix.T).toarray()
            nx_graph = nx.from_numpy_array(sim_mat)
            scores = nx.pagerank(nx_graph)
            top_indices = sorted(scores, key=scores.get, reverse=True)[:top_n]
            top_indices.sort()
            return " ".join([sentences[i] for i in top_indices])
        except: return " ".join(sentences[:top_n])

    # --- 3. LSA (잠재 의미 분석) 기반 요약 ---
    def lsa_summary(self, text, top_n=3):
        sentences = self.split_sentences(text)
        if len(sentences) <= top_n: return " ".join(sentences)
        sent_nouns = [self.get_nouns(s) for s in sentences]
        try:
            tfidf_matrix = self.vectorizer.fit_transform(sent_nouns)
            # 특이값 분해 (SVD)로 잠재 의미 추출
            svd = TruncatedSVD(n_components=1) # 가장 중요한 주제 1개 추출
            svd.fit(tfidf_matrix)
            # 각 문장이 해당 주제와 얼마나 관련있는지 확인
            scores = np.abs(svd.components_ @ tfidf_matrix.T.toarray()).flatten()
            top_indices = scores.argsort()[-top_n:][::-1]
            top_indices.sort()
            return " ".join([sentences[i] for i in top_indices])
        except: return " ".join(sentences[:top_n])

    # --- 4. LexRank 기반 요약 ---
    def lexrank_summary(self, text, top_n=3, threshold=0.1):
        sentences = self.split_sentences(text)
        if len(sentences) <= top_n: return " ".join(sentences)
        sent_nouns = [self.get_nouns(s) for s in sentences]
        try:
            tfidf_matrix = self.vectorizer.fit_transform(sent_nouns)
            sim_mat = cosine_similarity(tfidf_matrix)
            # 특정 임계값 이하의 유사도는 연결 끊기
            sim_mat[sim_mat < threshold] = 0
            # 정규화하여 인접 행렬로 변환 후 랭킹 계산
            nx_graph = nx.from_numpy_array(sim_mat)
            scores = nx.pagerank(nx_graph)
            top_indices = sorted(scores, key=scores.get, reverse=True)[:top_n]
            top_indices.sort()
            return " ".join([sentences[i] for i in top_indices])
        except: return " ".join(sentences[:top_n])

    # --- 5. MMR (중복 제거 강조) 기반 요약 ---
    def mmr_summary(self, text, top_n=3, lambda_val=0.5):
        sentences = self.split_sentences(text)
        if len(sentences) <= top_n: return " ".join(sentences)
        sent_nouns = [self.get_nouns(s) for s in sentences]
        try:
            tfidf_matrix = self.vectorizer.fit_transform(sent_nouns)
            scores = np.array(tfidf_matrix.sum(axis=1)).flatten()
            sim_mat = cosine_similarity(tfidf_matrix)
            
            summary_indices = [np.argmax(scores)] # 첫 번째 문장은 가장 높은 점수
            remaining_indices = [i for i in range(len(sentences)) if i not in summary_indices]
            
            while len(summary_indices) < top_n:
                mmr_values = []
                for i in remaining_indices:
                    # 중요도 가중치 - (이미 선택된 문장과의 최대 유사도 가중치)
                    relevance = scores[i]
                    redundancy = max([sim_mat[i][j] for j in summary_indices])
                    mmr_val = lambda_val * relevance - (1 - lambda_val) * redundancy
                    mmr_values.append((mmr_val, i))
                
                next_idx = max(mmr_values, key=lambda x: x[0])[1]
                summary_indices.append(next_idx)
                remaining_indices.remove(next_idx)
            
            summary_indices.sort()
            return " ".join([sentences[i] for i in summary_indices])
        except: return " ".join(sentences[:top_n])

# --- 메인 실행부 ---
def main():
    file_name = 'ai_times.csv'
    df = pd.read_csv(file_name)
    summarizer = SummaryMachine()
    
    print(f"총 {len(df)}개 기사에 대해 5가지 알고리즘 요약을 시작합니다...")

    df['TF-IDF'] = df['context'].apply(lambda x: summarizer.tfidf_summary(x, 3))
    df['TextRank'] = df['context'].apply(lambda x: summarizer.textrank_summary(x, 3))
    df['LSA'] = df['context'].apply(lambda x: summarizer.lsa_summary(x, 3))
    df['LexRank'] = df['context'].apply(lambda x: summarizer.lexrank_summary(x, 3))
    df['MMR'] = df['context'].apply(lambda x: summarizer.mmr_summary(x, 3))

    output_file = 'ai_times_results_final.csv'
    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"완료! 결과 저장: {output_file}")

if __name__ == "__main__":
    main()

총 5개 기사에 대해 5가지 알고리즘 요약을 시작합니다...
완료! 결과 저장: ai_times_results_final.csv


In [3]:
import pandas as pd

df1 = pd.read_csv('ai_times_results_final.csv')
df1

,title,context,TF-IDF,TextRank,LSA,LexRank,MMR
0,"KAIST, 분자 에너지 구조 예측 AI 개발…“신약·신소재 개발에 도움”","한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우...","한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우...","한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우...","한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우...","한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우...","한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우..."
1,"구글, 논문 일러스트 생성 AI ‘페이퍼바나나’ 공개...""환각 낮추고 미적 기준 올려""","AI가 참고 문헌 검색과 정리를 넘어, 학술 논문에 들어가는 이미지까지 처리하게 됐...",강한 원색보다 소프트한?파스텔?계열 색상을 선호하는 등?학계에 일반적인 패턴을 체계...,페이퍼바나나는 구글의 '나노바나나 프로' 등?최신 비전-언어 모델(VLM)과 이미지...,"특히, 시각화 에이전트는 '이미지 생성'과 '코드 생성'을 선택적으로 사용한다.?이...","2단계인 ‘반복 개선’에서는 시각화 에이전트와 비평 에이전트가 협력한다. 특히, 시...",구글은 베이징대학교 연구진과?6일(현지시간) 출판 수준의 학술 도식과 그래프를 자동...
2,29000TWh 시대…전력 시스템 전환의 열쇠는 무엇인가,국제에너지기구(IEA)가 6일 ‘Electricity 2026’ 행사를 통해 글로벌...,IEA는 2026년까지 세계 전력 소비가 사상 최고 수준인 약 29 000테라와트시...,행사는 세계적으로 전력 수요가 빠르게 증가하고 있는 시점에 맞춰 전력 시스템의 효율...,행사는 세계적으로 전력 수요가 빠르게 증가하고 있는 시점에 맞춰 전력 시스템의 효율...,행사는 세계적으로 전력 수요가 빠르게 증가하고 있는 시점에 맞춰 전력 시스템의 효율...,IEA는 2026년까지 세계 전력 소비가 사상 최고 수준인 약 29 000테라와트시...
3,산업용 RAG가 실패하는 이유 중 하나는 데이터 '청킹' 방식 때문,중공업 등 일부 산업 분야에서 검색 증강 생성(RAG)이 별 효과를 보지 못하는 결...,AI 아키텍트이자 데이터 엔지니어 디푸 쿠마르 싱 후지쓰 신기술 리더는 1일(현지시...,"이 때문에 ""RAG의 신뢰성을 향상하는 것은 더 큰 모델을 구입하는 것이 아니라, ...","이 때문에 ""RAG의 신뢰성을 향상하는 것은 더 큰 모델을 구입하는 것이 아니라, ...","이 때문에 ""RAG의 신뢰성을 향상하는 것은 더 큰 모델을 구입하는 것이 아니라, ...",AI 아키텍트이자 데이터 엔지니어 디푸 쿠마르 싱 후지쓰 신기술 리더는 1일(현지시...
4,"프랑스 연구팀, AI 심리·정서 추적 프레임워크 ‘DefMoN’ 공개",프랑스 라이언 리서치 연구소(RRI) 김상백 박사팀이 AI의 정서·심리 판단 근거를...,프랑스 라이언 리서치 연구소(RRI) 김상백 박사팀이 AI의 정서·심리 판단 근거를...,프랑스 라이언 리서치 연구소(RRI) 김상백 박사팀이 AI의 정서·심리 판단 근거를...,프랑스 라이언 리서치 연구소(RRI) 김상백 박사팀이 AI의 정서·심리 판단 근거를...,이번 프레임워크는?심리학?이론을?바탕으로?판단?과정을?추적하고?이를?재현한 합성 데...,프랑스 라이언 리서치 연구소(RRI) 김상백 박사팀이 AI의 정서·심리 판단 근거를...
